# 02 Cleaning

This notebook covers the cleaning and preprocessing of the extracted sample dataset.

## Key Cleaning Steps:
1. **Drop Unnecessary Columns:** Remove columns with 100% nulls (like `End_Lat`, `End_Lng`), single-value columns (like `Country`, `Turning_Loop`), and redundant or uninformative columns (`ID`, `Source`, `Description`).
2. **Handle Null Values intelligently:**
   - We avoid simple mean/median imputation which can distort distributions and destroy relationships.
   - Instead, we use **Relationship-based Imputation**: e.g., filling missing Temperature using the median for the specific State, Month, and Hour.
3. **Feature Engineering:** Create derived columns useful for Tableau and EDA, such as `Year`, `Month`, `Hour`, `Duration_min`, `Is_Weekend`, and `Time_of_Day`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.etl_pipeline import clean_accidents_data

In [2]:
EXTRACTED_PATH = PROJECT_ROOT / 'data/processed/extracted_sample.csv'
PROCESSED_PATH = PROJECT_ROOT / 'data/processed/cleaned_dataset.csv'

df = pd.read_csv(EXTRACTED_PATH)
print(f"Raw extracted shape: {df.shape}")

Raw extracted shape: (99998, 46)


### Applying the Cleaning Pipeline
The full logic is encapsulated in `scripts/etl_pipeline.py`. It executes the steps outlined above.

In [3]:
# The cleaning pipeline might output some warnings or logs about the imputation process
if PROCESSED_PATH.exists():
    print(f"Loading existing cleaned dataset from {PROCESSED_PATH}")
    clean_df = pd.read_csv(PROCESSED_PATH)
else:
    clean_df = clean_accidents_data(df)
    PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
    clean_df.to_csv(PROCESSED_PATH, index=False)
    print(f'Saved cleaned dataset to {PROCESSED_PATH}')

Loading existing cleaned dataset from /Users/mohitsingh/Desktop/DVA-capstone/Section-A_G12_DeliverIQ/data/processed/cleaned_dataset.csv


In [4]:
print(f"Cleaned dataset shape: {clean_df.shape}")
print(f"Total nulls remaining: {clean_df.isnull().sum().sum()}")

Cleaned dataset shape: (95607, 46)
Total nulls remaining: 0


In [5]:
clean_df.head()

,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,Distance(mi),Street,City,County,State,...,Month,Day_of_Week,Day_Name,Hour,Is_Weekend,Time_of_Day,Season,Duration_min,Road_Feature_Count,Is_Rush_Hour
0,4,2017-02-25 01:21:34,2017-02-25 07:21:34,29.949330,-95.291820,1.718,US-59 S,Humble,Harris,TX,...,2.0,5.0,Saturday,1.0,True,Night,Winter,360.000000,0,False
1,4,2022-10-13 23:14:46,2022-10-14 01:15:36,35.037490,-89.795978,0.371,Bill Morris Pkwy,Memphis,Shelby,TN,...,10.0,3.0,Thursday,23.0,False,Night,Fall,120.833333,1,False
2,3,2018-12-03 05:54:08,2018-12-03 06:23:59,37.824001,-122.316925,0.000,I-80 W,Oakland,Alameda,CA,...,12.0,0.0,Monday,5.0,False,Morning,Winter,29.850000,1,False
3,4,2019-06-24 16:18:39,2019-06-24 16:47:09,44.477478,-88.470745,0.407,W5590 State Highway 54,Black Creek,Outagamie,WI,...,6.0,0.0,Monday,16.0,False,Afternoon,Summer,28.500000,0,True
4,2,2021-06-13 17:43:00,2021-06-13 18:46:00,33.956033,-118.109349,1.411,I-5 S,Downey,Los Angeles,CA,...,6.0,6.0,Sunday,17.0,True,Evening,Summer,63.000000,1,True
